***Summary, Translation, and Generation***

In this part of the project, the reviews are transformed into cleaner and more usable text versions.

The goal here is to improve the quality of the dataset by creating corrected and standardized text columns, and when needed, translated versions of the reviews.

 This step makes the data easier to analyze and prepares it for the next NLP tasks such as topic detection, sentiment analysis, and supervised learning.

In [2]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm

tqdm.pandas()


In [3]:
df = pd.read_csv("cleaned_dataset.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (34428, 11)


,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_clean,avis_cor
0,4.0,audurier-c-136272,La personne au téléphone était Clair et sympat...,L'olivier Assurance,auto,train,2021-10-06,2021-10-01,The person on the phone was clear and friendly...,la personne au téléphone était clair et sympat...,la personne au téléphone était clair et sympat...
1,4.0,paul-a-122970,"Satisfait.\n\nRéactivité, simplicité. Prix att...",APRIL Moto,moto,train,2021-07-09,2021-07-01,"Satisfied.\n\nReactivity, simplicity. Attracti...",satisfait réactivité simplicité prix attractif...,satisfait réactivité simplicité prix attractif...
2,1.0,kitty-38517,"Assureur à fuir, n assure pas ses responsabili...",SwissLife,vie,train,2020-10-15,2020-10-01,"Insurer to flee, does not ensure its responsib...",assureur à fuir n assure pas ses responsabilit...,assureur à fuir un assure pas ses responsabili...
3,1.0,laure97134-87907,Voilà 3 mois que la GMF me fait attendre pour ...,GMF,habitation,train,2020-03-03,2020-03-01,The GMF has been waiting for a water damage fo...,voilà mois que la gmf me fait attendre pour un...,voilà mois que la me me fait attendre pour un ...
4,3.0,bourouane-l-129916,Je suis bien avec cet assurance.elle est prati...,L'olivier Assurance,auto,train,2021-08-28,2021-08-01,I am good with this insurance. She is practica...,je suis bien avec cet assurance elle est prati...,je suis bien avec cet assurance elle est prati...


## Step 1 — Translation
Since our dataset already contains English translations in the `avis_en` column,
we will use the corrected French reviews (`avis_cor`) and translate them to English
as well. This gives us a cleaner English version based on the corrected text,
which will be more useful for English-based NLP models.
We use the `deep-translator` library which uses Google Translate under the hood.
Since translating 34,000+ rows would take too long, we apply it to a representative
sample of 500 reviews.

In [4]:
def translate_to_english(text):
    if pd.isna(text) or str(text).strip() == "":
        return text
    try:
        return GoogleTranslator(source="fr", target="en").translate(str(text))
    except:
        return text

# Apply translation on a sample of 500 reviews
sample = df.sample(500, random_state=42).copy()
sample["avis_cor_en"] = sample["avis_cor"].progress_apply(translate_to_english)

print("Translation complete!")
sample[["avis_cor", "avis_cor_en"]].head(5)

  0%|          | 0/500 [00:00<?, ?it/s]

Translation complete!


,avis_cor,avis_cor_en
33636,dommage de toujours payer un frais de dossier ...,a shame to always pay an administrative fee on...
18072,très satisfaite du service très facile à gérer...,very satisfied with the service very easy to m...
9724,assuré chez cette entreprise je ai subi un acc...,insured with this company I suffered a car acc...
31011,les prix réduits et des garanties plus étendue...,reduced prices and more extensive guarantees w...
18239,je suis satisfait du service les prix me convi...,I am satisfied with the service the prices sui...


## Step 2 — Summary Generation
In this step we generate short summaries of the reviews using a pre-trained
summarization model from HuggingFace. Summarization is useful for our Streamlit
application where we want to display a quick overview of long reviews.
We use the `facebook/bart-large-cnn` model which is one of the best performing
summarization models available.

In [5]:

import subprocess
subprocess.run(["pip", "install", "transformers", "torch"])


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', 'transformers', 'torch'], returncode=0)

In [6]:
from transformers import pipeline

summarizer = pipeline("summarization", model="t5-small")

def summarize_text(text):
    if pd.isna(text) or len(str(text).split()) < 30:
        return text  # too short to summarize
    try:
        result = summarizer(str(text), max_length=60, min_length=20, do_sample=False)
        return result[0]['summary_text']
    except:
        return text

# Apply on the same 500 sample using English reviews
sample["avis_summary"] = sample["avis_en"].progress_apply(summarize_text)
sample[["avis_en", "avis_summary"]].head(5)

/Users/dovila/nlp_project/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  0%|          | 0/500 [00:00<?, ?it/s]

Your max_length is set to 60, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
Your max_length is set to 60, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
Your max_length is set to 60, but your input_length is only 53. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)
Your max_length is set to 60, but your input_length is only 48. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=24)
Your max

,avis_en,avis_summary
33636,Too bad to always pay a file fees on each new ...,Too bad to always pay a file fees on each new ...
18072,Very satisfied with the service very easy to m...,very satisfied with the service very easy to m...
9724,Insured with this company. I underwent a non -...,the company mandated an expert who estimated t...
31011,Reduced prices and more extensive guarantees w...,Reduced prices and more extensive guarantees w...
18239,I am satisfied with the service. The prices su...,I am satisfied with the service. the prices su...


## Step 3 — Saving the Enriched Dataset
We now save the enriched dataset containing all the cleaned, translated,
and summarized columns. This file will be used in all subsequent notebooks.

In [8]:
# Add the translated and summarized columns back to the full dataframe
df["avis_cor_en"] = None
df["avis_summary"] = None

df.loc[sample.index, "avis_cor_en"] = sample["avis_cor_en"]
df.loc[sample.index, "avis_summary"] = sample["avis_summary"]

# Save the enriched dataset
df.to_csv("enriched_dataset.csv", index=False)
print(f"Final shape: {df.shape}")
df.head()

Final shape: (34428, 13)


,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_clean,avis_cor,avis_cor_en,avis_summary
0,4.0,audurier-c-136272,La personne au téléphone était Clair et sympat...,L'olivier Assurance,auto,train,2021-10-06,2021-10-01,The person on the phone was clear and friendly...,la personne au téléphone était clair et sympat...,la personne au téléphone était clair et sympat...,None,None
1,4.0,paul-a-122970,"Satisfait.\n\nRéactivité, simplicité. Prix att...",APRIL Moto,moto,train,2021-07-09,2021-07-01,"Satisfied.\n\nReactivity, simplicity. Attracti...",satisfait réactivité simplicité prix attractif...,satisfait réactivité simplicité prix attractif...,None,None
2,1.0,kitty-38517,"Assureur à fuir, n assure pas ses responsabili...",SwissLife,vie,train,2020-10-15,2020-10-01,"Insurer to flee, does not ensure its responsib...",assureur à fuir n assure pas ses responsabilit...,assureur à fuir un assure pas ses responsabili...,None,None
3,1.0,laure97134-87907,Voilà 3 mois que la GMF me fait attendre pour ...,GMF,habitation,train,2020-03-03,2020-03-01,The GMF has been waiting for a water damage fo...,voilà mois que la gmf me fait attendre pour un...,voilà mois que la me me fait attendre pour un ...,None,None
4,3.0,bourouane-l-129916,Je suis bien avec cet assurance.elle est prati...,L'olivier Assurance,auto,train,2021-08-28,2021-08-01,I am good with this insurance. She is practica...,je suis bien avec cet assurance elle est prati...,je suis bien avec cet assurance elle est prati...,None,None
